# Electronics — the op-amp, and the five places the abstraction leaks

The previous notebook ended with three stages that were each useless alone: a differential pair that rejects common mode but has awkward output impedance, a gain stage that loads badly at both ends, and a follower with no gain. Wire them in that order and you have an operational amplifier — differential input, very high gain, low output impedance.

What makes it worth a chapter of its own is that the internal detail then stops mattering. With enough loop gain the closed-loop behaviour depends only on the feedback network:

$$A_{cl}=\frac{A}{1+A\beta}\;\longrightarrow\;\frac{1}{\beta}$$

so the circuit is designed with two resistors and the device becomes an abstraction: **infinite gain, infinite input impedance, zero output impedance, and the two inputs at the same voltage**.

That abstraction is extremely good and it leaks in exactly five places — finite gain, finite bandwidth, slew rate, DC errors, and stability. Each has its own section, each is measured rather than asserted, and each is the reason a real design fails when the ideal one said it would work.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "electronics · op-amps", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


VT = 0.02585          # kT/q at 300 K
IS_BJT = 1e-15
BETA = 150.0
VA = 50.0             # Early voltage
VTH_N = 1.0           # MOSFET threshold
KN = 2e-3             # MOSFET transconductance parameter, A/V^2
LAMBDA = 0.02


def bjt_ic(vbe, vce=5.0, Is=IS_BJT, va=VA):
    """Forward-active collector current with the Early effect."""
    ic = Is * np.exp(np.clip(vbe, -2, 1.2) / VT)
    return ic * (1 + np.maximum(vce, 0) / va)


def mos_id(vgs, vds, vth=VTH_N, k=KN, lam=LAMBDA):
    """Square-law NMOS: cutoff, triode, saturation."""
    vgs = np.asarray(vgs, float); vds = np.asarray(vds, float)
    vov = vgs - vth
    tri = k * (vov * vds - 0.5 * vds ** 2)
    sat = 0.5 * k * vov ** 2 * (1 + lam * vds)
    out = np.where(vds < vov, tri, sat)
    return np.where(vov <= 0, 0.0, np.maximum(out, 0.0))


def mos_region(vgs, vds, vth=VTH_N):
    if vgs - vth <= 0:
        return "cutoff"
    return "triode" if vds < vgs - vth else "saturation"


def transistor_npn(ax, p, v, vmax, label=None, s=0.42, flip=False):
    """NPN symbol: base left, collector up, emitter down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.55, p[0] - s * 0.55], [p[1] - s, p[1] + s],
            color=FG, lw=2.6, zorder=3)
    ax.plot([p[0] - s * 1.5, p[0] - s * 0.55], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] + s * 0.45,
            p[1] + s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] - s * 0.45,
            p[1] - s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.2, p[1] - s * 0.82], [p[0] + s * 0.05, p[1] - s * 0.45],
         [p[0] + s * 0.5, p[1] - s * 0.62]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.05, p[1], label, color=FG, fontsize=8,
                va="center")


def transistor_nmos(ax, p, v, vmax, label=None, s=0.42):
    """NMOS symbol: gate left, drain up, source down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.95, p[0] - s * 0.95], [p[1] - s, p[1] + s],
            color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 1.9, p[0] - s * 0.95], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    for dy in (-1, 0, 1):
        y0 = p[1] + dy * s * 0.62
        ax.plot([p[0] - s * 0.5, p[0] - s * 0.5],
                [y0 - s * 0.28, y0 + s * 0.28], color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1], p[1]], color=FG,
            lw=2.0, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.35, p[1]], [p[0] + s * 0.05, p[1] + s * 0.2],
         [p[0] + s * 0.05, p[1] - s * 0.2]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.25, p[1], label, color=FG, fontsize=8,
                va="center")


def db(x):
    return 20 * np.log10(np.maximum(np.abs(x), 1e-14))


def opamp_tri(ax, p, w=1.5, h=1.4, label=None, invert_top=True):
    """Standard op-amp triangle with labelled inputs."""
    p = np.asarray(p, float)
    ax.add_patch(mpatches.Polygon(
        [[p[0], p[1] - h / 2], [p[0], p[1] + h / 2], [p[0] + w, p[1]]],
        closed=True, fill=False, ec=FG, lw=2.2, zorder=4))
    yt, yb = p[1] + h * 0.24, p[1] - h * 0.24
    ax.text(p[0] + 0.16, yt, "−" if invert_top else "+", color=FG,
            fontsize=11, ha="center", va="center", zorder=5)
    ax.text(p[0] + 0.16, yb, "+" if invert_top else "−", color=FG,
            fontsize=11, ha="center", va="center", zorder=5)
    if label:
        ax.text(p[0] + w * 0.42, p[1] - h * 0.62, label, color=MUTED,
                fontsize=7.5, ha="center")
    return (p[0], yt), (p[0], yb), (p[0] + w, p[1])


def two_pole_loop(A0, f1, f2, beta, f):
    L = A0 * beta / ((1 + 1j * f / f1) * (1 + 1j * f / f2))
    return L


def phase_margin(A0, f1, f2, beta):
    f = np.logspace(-2, 10, 40001)
    L = two_pole_loop(A0, f1, f2, beta, f)
    i = int(np.argmin(np.abs(np.abs(L) - 1)))
    return 180 + np.degrees(np.angle(L[i])), f[i]


def closed_step(A0, f1, f2, beta, n=3000):
    """Exact step response of the closed loop (second order)."""
    w1, w2 = 2 * np.pi * f1, 2 * np.pi * f2
    wn = np.sqrt(w1 * w2 * (1 + A0 * beta))
    zeta = (w1 + w2) / (2 * wn)
    T = 12 / (zeta * wn)
    t = np.linspace(0, T, n)
    dc = A0 * beta / (1 + A0 * beta)
    if zeta < 1:
        wd = wn * np.sqrt(1 - zeta ** 2)
        y = dc * (1 - np.exp(-zeta * wn * t)
                  * (np.cos(wd * t)
                     + zeta / np.sqrt(1 - zeta ** 2) * np.sin(wd * t)))
    else:
        s = np.sqrt(zeta ** 2 - 1)
        s1, s2 = (-zeta + s) * wn, (-zeta - s) * wn
        y = dc * (1 + (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s1 - s2))
    return t, y, zeta, wn, dc


print("op-amp engine ready")
print("Acl = A/(1+Aβ)   error ≈ −1/(1+Aβ)   BW = GBW/gain")
print(f"VT = {VT*1e3:.2f} mV   e-fold per VT   decade per {VT*np.log(10)*1e3:.2f} mV")
print(f"MOSFET Vth = {VTH_N:.2f} V   k = {KN*1e3:.2f} mA/V^2   VA = {VA:.0f} V")

op-amp engine ready
Acl = A/(1+Aβ)   error ≈ −1/(1+Aβ)   BW = GBW/gain
VT = 25.85 mV   e-fold per VT   decade per 59.52 mV
MOSFET Vth = 1.00 V   k = 2.00 mA/V^2   VA = 50 V


## The virtual short — why two resistors are enough

If the gain is enormous and the output is finite, the input difference must be almost zero:

$$v_+-v_-=\frac{v_{out}}{A}\;\to\;0$$

That single statement, plus "no current flows into the inputs", solves every op-amp circuit without any device analysis at all. The inverting node sits at ground without being connected to it — a **virtual ground** — so the input current is $v_{in}/R_1$, all of it flows on through $R_2$, and

$$A_v=-\frac{R_2}{R_1}
\qquad\qquad
A_v=1+\frac{R_2}{R_1}\ \text{(non-inverting)}$$

The two configurations differ by exactly 1, which is not a coincidence: the non-inverting version also passes the input straight through, so it gets the inverting gain plus unity. It cannot produce a gain below 1; the inverting version can.

The panels trace the current through the network so the virtual-short argument is visible rather than asserted. Note the asymmetry in input impedance — the inverting configuration presents $R_1$, so it *loads its source*, while the non-inverting one presents the op-amp's own input impedance, which is essentially infinite. That is often the deciding factor between them.

In [2]:
CONFIGS = ["inverting", "non-inverting", "follower", "difference"]


def config_gain(cfg, R1, R2):
    if cfg == "inverting":
        return -R2 / R1, R1
    if cfg == "non-inverting":
        return 1 + R2 / R1, np.inf
    if cfg == "follower":
        return 1.0, np.inf
    return R2 / R1, R1


def draw_ideal(k, cfg, R1_k, R2_k, vin):
    R1, R2 = R1_k * 1e3, R2_k * 1e3
    Av, Rin = config_gain(cfg, R1, R2)
    vout = np.clip(Av * vin, -12, 12)
    icur = abs(vin) / R1 if cfg in ("inverting", "difference") else 0.0
    t = np.linspace(0, 2, 400)
    sig = vin * np.sin(2 * np.pi * t)
    out = np.clip(Av * sig, -12, 12)
    kk = int(min(k, len(t) - 1))
    vmax = 12.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 5.4), (-1.2, 2.6))
    (pn, pp, po) = opamp_tri(a0, (2.4, 0.6), 1.5, 1.4)
    vmin_node = 0.0 if cfg != "non-inverting" else sig[kk]
    if cfg == "inverting":
        wire(a0, [(0.3, 1.0), (1.0, 1.0)], sig[kk], vmax)
        resistor(a0, (1.0, 1.0), (2.0, 1.0), sig[kk] / 2, vmax, f"R1 {R1_k:.1f}k")
        wire(a0, [(2.0, 1.0), pn], 0.0, vmax)
        wire(a0, [pn, (2.0, 2.2)], 0.0, vmax)
        resistor(a0, (2.0, 2.2), (4.4, 2.2), 0.0, vmax, f"R2 {R2_k:.1f}k")
        wire(a0, [(4.4, 2.2), (4.4, 0.6), po], out[kk], vmax)
        wire(a0, [pp, (1.6, 0.2), (1.6, -0.6)], 0.0, vmax)
        a0.text(1.9, 1.22, "virtual gnd", color=DOT, fontsize=7)
        node_dot(a0, (2.0, 1.0), 0.0, vmax)
    elif cfg == "non-inverting":
        wire(a0, [(0.3, 0.2), pp], sig[kk], vmax)
        wire(a0, [pn, (1.9, 1.0), (1.9, 2.2)], sig[kk], vmax)
        resistor(a0, (1.9, 2.2), (4.4, 2.2), sig[kk] / 2, vmax, f"R2 {R2_k:.1f}k")
        wire(a0, [(4.4, 2.2), (4.4, 0.6), po], out[kk], vmax)
        resistor(a0, (1.9, 1.0), (1.9, -0.4), sig[kk] / 2, vmax, f"R1 {R1_k:.1f}k")
        wire(a0, [(1.9, -0.4), (1.9, -0.9), (0.6, -0.9)], 0.0, vmax)
        node_dot(a0, (1.9, 1.0), sig[kk], vmax)
    elif cfg == "follower":
        wire(a0, [(0.3, 0.2), pp], sig[kk], vmax)
        wire(a0, [pn, (2.0, 1.0), (2.0, 2.2)], out[kk], vmax)
        wire(a0, [(2.0, 2.2), (4.4, 2.2), (4.4, 0.6), po], out[kk], vmax)
    else:
        wire(a0, [(0.3, 1.0), (1.0, 1.0)], sig[kk], vmax)
        resistor(a0, (1.0, 1.0), (2.0, 1.0), sig[kk] / 2, vmax, "R1")
        wire(a0, [(2.0, 1.0), pn], 0.0, vmax)
        wire(a0, [pn, (2.0, 2.2)], 0.0, vmax)
        resistor(a0, (2.0, 2.2), (4.4, 2.2), 0.0, vmax, "R2")
        wire(a0, [(4.4, 2.2), (4.4, 0.6), po], out[kk], vmax)
        wire(a0, [(0.3, -0.2), (1.0, -0.2)], 0.0, vmax)
        resistor(a0, (1.0, -0.2), (1.8, -0.2), 0.0, vmax, "R1")
        wire(a0, [(1.8, -0.2), pp], 0.0, vmax)
        resistor(a0, (1.8, -0.2), (1.8, -1.0), 0.0, vmax, "R2")
    wire(a0, [po, (5.0, 0.6)], out[kk], vmax)
    node_dot(a0, (5.0, 0.6), out[kk], vmax)
    a0.text(5.05, 0.85, "out", color=FG, fontsize=8)
    if icur > 0:
        charge_dots(a0, seg((1.0, 1.0), (2.0, 1.0), 18), k / 100 * icur * 3e5,
                    spacing=0.20, ms=3.2)
        charge_dots(a0, seg((2.0, 2.2), (4.4, 2.2), 26), k / 100 * icur * 3e5,
                    spacing=0.20, ms=3.2)
    a0.set_title(f"{cfg} — $A_v$ = {Av:+.4f}", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t, sig, color=POS, lw=1.5, label="in")
    a1.plot(t, out, color=DOT, lw=1.6, label="out")
    a1.axhline(0, color=GRIDC, lw=0.8)
    a1.axvline(t[kk], color=FG, lw=1.0, ls=":")
    a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title("the same current flows through both resistors")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    rr = np.logspace(-1, 3, 300)
    a2.loglog(rr, rr / R1_k, color=ORANGE, lw=1.7, label="|inverting|  R2/R1")
    a2.loglog(rr, 1 + rr / R1_k, color=POS, lw=1.7, label="non-inverting  1+R2/R1")
    a2.axhline(1, color=MUTED, lw=0.8, ls=":")
    a2.axvline(R2_k, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("R2  (kΩ)"); a2.set_ylabel("|gain|"); a2.legend(fontsize=7)
    a2.set_title("they differ by exactly 1 — only inverting goes below unity")

    readout(fig, 0.845, 0.90, [
        "CONFIGURATION", "─" * 26,
        f"{cfg:>26s}",
        f"R1          {R1_k:>10.2f}kΩ",
        f"R2          {R2_k:>10.2f}kΩ",
        f"R2/R1       {R2/R1:>10.4f}",
        "", "IDEAL RESULT", "─" * 26,
        f"Av          {Av:>+10.4f}",
        f"in dB       {20*np.log10(abs(Av)):>10.3f}dB",
        f"Rin         {(Rin/1e3 if np.isfinite(Rin) else np.inf):>10.2f}kΩ",
        f"Rout        {0.0:>10.2f}Ω",
        "", "NOW", "─" * 26,
        f"vin         {sig[kk]:>+10.4f}V",
        f"vout        {out[kk]:>+10.4f}V",
        f"v+ − v−     {0.0:>10.6f}V",
        f"current     {icur*1e6:>10.3f}µA",
        "", "the virtual short is",
        "the only assumption",
        "you need",
    ])
    footer(fig, "v+ = v−  and  no input current   ·   "
                "inverting −R2/R1   ·   non-inverting 1+R2/R1")
    plt.show()


_p1, _s1 = timeline(99, step=2)
w1 = dict(cfg=widgets.Dropdown(options=CONFIGS, value="inverting",
                               description="topology:", **SL),
          R1_k=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="R1 (kΩ):", **SL),
          R2_k=widgets.FloatSlider(value=10, min=0.1, max=100, step=0.1,
                                   description="R2 (kΩ):", **SL),
          vin=widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                                  description="input (V):", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["cfg"], w1["R1_k"], w1["R2_k"]]),
                      widgets.HBox([w1["vin"], _p1, _s1])]),
        widgets.interactive_output(draw_ideal, w1))

Output()

## Leak one — the gain is not infinite

The virtual short assumed $v_+-v_-\to0$. With finite $A$ it is $v_{out}/A$, and the closed-loop gain falls short of the ideal by a factor that depends only on the **loop gain** $A\beta$:

$$\frac{A_{cl}-1/\beta}{1/\beta}=-\frac{1}{1+A\beta}$$

The panel confirms this identity to eight decimal places at every gain tested. What it means in practice is a simple budget: for a closed-loop gain of 11, an op-amp with $A=10^3$ gives $-1.088\%$ error, $A=10^4$ gives $-0.110\%$, $A=10^5$ gives $-0.011\%$. Every decade of open-loop gain buys one decade of accuracy.

The uncomfortable part is the coupling. The loop gain is $A\beta$, and $\beta$ is small when the closed-loop gain is large — so a circuit configured for a gain of 1000 has a thousand times less loop gain than a follower built from the same part, and correspondingly worse accuracy, linearity, output impedance and bandwidth. **High closed-loop gain is expensive in every dimension at once**, which is why gain is usually distributed across several stages rather than demanded from one.

Drag the open-loop gain down and watch the error grow. Below about 60 dB the "ideal" formula is no longer a useful description of the circuit.

In [ ]:
def draw_finite(A_dB, R1_k, R2_k, cfg):
    A = 10 ** (A_dB / 20)
    R1, R2 = R1_k * 1e3, R2_k * 1e3
    beta = R1 / (R1 + R2)
    ideal = (1 + R2 / R1) if cfg == "non-inverting" else -R2 / R1
    if cfg == "non-inverting":
        acl = A / (1 + A * beta)
    else:
        acl = -(R2 / R1) * (A * beta) / (1 + A * beta)
    err = (abs(acl) - abs(ideal)) / abs(ideal) * 100
    loop = A * beta
    AA = np.logspace(1, 8, 400)
    errs = -1 / (1 + AA * beta) * 100

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.semilogx(AA, errs, color=POS, lw=1.9)
    a0.axvline(A, color=FG, lw=1.0, ls="--")
    a0.plot([A], [-1 / (1 + loop) * 100], "o", ms=8, color=DOT)
    for lvl, lab in ((-1, "1%"), (-0.1, "0.1%"), (-0.01, "0.01%")):
        a0.axhline(lvl, color=GRIDC, lw=0.7)
        a0.text(AA[2], lvl * 1.15, lab, color=MUTED, fontsize=7)
    a0.set_ylim(-12, 0.5)
    a0.set_xlabel("open-loop gain  A"); a0.set_ylabel("gain error  (%)")
    a0.set_title(f"error = −1/(1+Aβ)  ·  loop gain here {loop:,.1f}")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    gains = np.logspace(0, 3.3, 200)
    for Adb in (60, 80, 100, 120):
        Ax = 10 ** (Adb / 20)
        b = 1 / gains
        a1.loglog(gains, np.abs(1 / (1 + Ax * b)) * 100,
                  color=POS if abs(Adb - A_dB) < 3 else MUTED,
                  lw=1.8 if abs(Adb - A_dB) < 3 else 0.9,
                  alpha=1.0 if abs(Adb - A_dB) < 3 else 0.5)
        a1.text(gains[-1] * 1.05, abs(1 / (1 + Ax / gains[-1])) * 100,
                f"{Adb}dB", color=MUTED, fontsize=6.5, va="center")
    a1.axvline(abs(ideal), color=FG, lw=1.0, ls="--")
    a1.set_xlabel("closed-loop gain"); a1.set_ylabel("error  (%)")
    a1.set_title("more closed-loop gain means less loop gain, so worse everything")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.bar([0, 1], [abs(ideal), abs(acl)], color=[MUTED, DOT], width=0.5)
    for x, v in ((0, abs(ideal)), (1, abs(acl))):
        a2.text(x, v, f"  {v:.5f}", ha="center", va="bottom", fontsize=8.5)
    a2.set_xticks([0, 1]); a2.set_xticklabels(["ideal 1/β", "actual"], fontsize=8)
    a2.set_ylim(0, abs(ideal) * 1.25)
    a2.set_title(f"shortfall {abs(err):.4f}%")

    readout(fig, 0.845, 0.88, [
        "CIRCUIT", "─" * 26,
        f"{cfg:>26s}",
        f"R1          {R1_k:>10.2f}kΩ",
        f"R2          {R2_k:>10.2f}kΩ",
        f"β = R1/(R1+R2){beta:>8.5f}",
        "", "AMPLIFIER", "─" * 26,
        f"A           {A:>10.4e}",
        f"in dB       {A_dB:>10.1f}dB",
        f"loop gain   {loop:>10.2f}",
        f"in dB       {20*np.log10(loop):>10.2f}dB",
        "", "RESULT", "─" * 26,
        f"ideal       {ideal:>+10.5f}",
        f"actual      {acl:>+10.5f}",
        f"error       {err:>+10.5f}%",
        f"−1/(1+Aβ)   {-1/(1+loop)*100:>+10.5f}%",
        "", "one decade of A buys",
        "one decade of accuracy",
    ], color=GREEN if abs(err) < 0.1 else ORANGE)
    footer(fig, f"Acl = A/(1+Aβ)   ·   fractional error exactly −1/(1+Aβ)   ·   "
                f"loop gain {loop:,.0f}")
    plt.show()


w2 = dict(A_dB=widgets.FloatSlider(value=100, min=20, max=160, step=5,
                                   description="open-loop dB:", **SL),
          R1_k=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="R1 (kΩ):", **SL),
          R2_k=widgets.FloatSlider(value=10, min=0.1, max=200, step=0.5,
                                   description="R2 (kΩ):", **SL),
          cfg=widgets.Dropdown(options=["non-inverting", "inverting"],
                               value="non-inverting", description="topology:", **SL))
display(widgets.HBox([w2["A_dB"], w2["R1_k"], w2["R2_k"], w2["cfg"]]),
        widgets.interactive_output(draw_finite, w2))

## Leaks two and three — bandwidth and slew rate are different limits

An op-amp is deliberately given a single dominant pole, so its gain falls at 20 dB/decade and the product of gain and bandwidth is constant:

$$\text{GBW}=A_0f_p,\qquad f_{-3\text{dB}}=\frac{\text{GBW}}{A_{cl}}$$

A 1 MHz part gives 1 MHz as a follower, 100 kHz at a gain of 10, and 1 kHz at a gain of 1000 — verified against the full expression $f_p(1+A_0\beta)$, which agrees to within 1%.

**Slew rate is not this.** It is a hard limit on $dv/dt$ set by a fixed internal current charging the compensation capacitor, $SR=I/C_c$, and it has nothing to do with frequency response. The two produce completely different symptoms and the square-wave panel separates them: small steps show an exponential rise governed by GBW, large steps show a **straight ramp** whose slope does not change with amplitude.

The crossover is where the sine wave's maximum slope $2\pi fV_{pk}$ exceeds $SR$, giving the full-power bandwidth:

$$f_{max}=\frac{SR}{2\pi V_{pk}}$$

At 0.5 V/µs that is 796 kHz for a 100 mV signal but only **15.9 kHz** for 5 V. An amplifier can be perfectly linear at small signal and grossly distorted at large signal *at the same frequency* — which is why a datasheet quotes both numbers, and why slew-limited distortion is triangular rather than clipped.

In [ ]:
def draw_gbw_slew(GBW_MHz, SR_Vus, gain, Vpk, f_kHz, wave):
    GBW, SR = GBW_MHz * 1e6, SR_Vus * 1e6
    f = f_kHz * 1e3
    bw = GBW / gain
    fmax = SR / (2 * np.pi * Vpk)
    fs = max(200 * f, 40 * bw)
    n = 4000
    t = np.arange(n) / fs
    if wave == "sine":
        drive = Vpk * np.sin(2 * np.pi * f * t)
    else:
        drive = Vpk * np.sign(np.sin(2 * np.pi * f * t))
    y = np.zeros(n)
    tau = 1 / (2 * np.pi * bw)
    for i in range(1, n):
        want = (drive[i] - y[i - 1]) / tau
        rate = np.clip(want, -SR, SR)
        y[i] = y[i - 1] + rate / fs
    lin = np.zeros(n)
    for i in range(1, n):
        lin[i] = lin[i - 1] + (drive[i] - lin[i - 1]) / tau / fs
    slewing = np.mean(np.abs(np.gradient(y, t)) > 0.98 * SR) * 100

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.35, 1.15, 0.55],
                          wspace=0.3, hspace=0.46, left=0.055, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot(t * 1e6, drive, color=MUTED, lw=1.0, ls="--", label="input")
    a0.plot(t * 1e6, lin, color=PURP, lw=1.2, label="bandwidth only")
    a0.plot(t * 1e6, y, color=DOT, lw=1.6, label="with slew limit")
    a0.set_xlim(0, min(3 / f * 1e6, t[-1] * 1e6))
    a0.set_xlabel("time  (µs)"); a0.set_ylabel("volts"); a0.legend(fontsize=7)
    a0.set_title(f"{'straight ramps = slewing' if slewing > 5 else 'exponential = bandwidth limited'}")

    a1 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    ff = np.logspace(1, 8, 400)
    a1.loglog(ff, GBW / ff, color=MUTED, lw=1.2, ls="--", label="open loop")
    a1.loglog(ff, np.full_like(ff, gain) / np.sqrt(1 + (ff / bw) ** 2),
              color=POS, lw=1.8, label=f"closed loop, gain {gain:.0f}")
    a1.axvline(bw, color=POS, lw=0.9, ls=":")
    a1.axvline(fmax, color=NEG, lw=1.2, ls="--")
    a1.text(fmax * 1.1, gain * 0.5, "full-power limit", color=NEG, fontsize=7)
    a1.axvline(f, color=FG, lw=1.0, ls="-")
    a1.set_ylim(0.1, max(gain * 2, 10))
    a1.set_xlabel("frequency  (Hz)"); a1.set_ylabel("gain")
    a1.legend(fontsize=7)
    a1.set_title(f"small-signal BW {bw/1e3:.2f} kHz  ·  full power only to "
                 f"{fmax/1e3:.2f} kHz")

    a2 = panel(fig.add_subplot(gs[:, 1]), GREEN)
    vp = np.logspace(-2, 1.2, 300)
    a2.loglog(vp, SR / (2 * np.pi * vp) / 1e3, color=GREEN, lw=1.8,
              label="slew limit")
    a2.axhline(bw / 1e3, color=POS, lw=1.4, ls="--", label="small-signal BW")
    a2.axvline(Vpk, color=FG, lw=1.0, ls=":")
    a2.plot([Vpk], [fmax / 1e3], "o", ms=7, color=DOT)
    a2.set_xlabel("output amplitude  (V peak)")
    a2.set_ylabel("usable frequency  (kHz)")
    a2.legend(fontsize=7)
    a2.set_title("bandwidth does not depend on amplitude — slew rate does")

    readout(fig, 0.845, 0.90, [
        "AMPLIFIER", "─" * 26,
        f"GBW         {GBW_MHz:>10.3f}MHz",
        f"slew rate   {SR_Vus:>10.3f}V/µs",
        f"closed gain {gain:>10.1f}",
        "", "BANDWIDTH", "─" * 26,
        f"BW = GBW/G  {bw/1e3:>10.3f}kHz",
        f"rise 0.35/BW{0.35/bw*1e6:>10.4f}µs",
        "", "SLEW", "─" * 26,
        f"amplitude   {Vpk:>10.3f}V",
        f"f max power {fmax/1e3:>10.3f}kHz",
        f"max dv/dt   {2*np.pi*f*Vpk/1e6:>10.4f}V/µs",
        f"available   {SR_Vus:>10.4f}V/µs",
        f"slewing     {slewing:>10.1f}% of time",
        "", "DRIVE", "─" * 26,
        f"frequency   {f_kHz:>10.2f}kHz",
        f"waveform    {wave:>14s}",
        "SLEW LIMITED" if slewing > 5 else "linear",
        "", "same part, same",
        "frequency, different",
        "amplitude → different",
        "failure",
    ], color=NEG if slewing > 5 else GREEN)
    footer(fig, f"BW = GBW/gain = {bw/1e3:.2f} kHz   ·   "
                f"f_full-power = SR/(2πV) = {fmax/1e3:.2f} kHz   ·   "
                f"two independent limits")
    plt.show()


w3 = dict(GBW_MHz=widgets.FloatSlider(value=1.0, min=0.1, max=20, step=0.1,
                                      description="GBW (MHz):", **SL),
          SR_Vus=widgets.FloatSlider(value=0.5, min=0.05, max=20, step=0.05,
                                     description="slew (V/µs):", **SL),
          gain=widgets.FloatSlider(value=1, min=1, max=100, step=1,
                                   description="closed gain:", **SL),
          Vpk=widgets.FloatSlider(value=1.0, min=0.05, max=10, step=0.05,
                                  description="amplitude (V):", **SL),
          f_kHz=widgets.FloatSlider(value=20, min=1, max=500, step=1,
                                    description="frequency kHz:", **SL),
          wave=widgets.Dropdown(options=["sine", "square"], value="sine",
                                description="waveform:", **SL))
display(widgets.VBox([widgets.HBox([w3["GBW_MHz"], w3["SR_Vus"], w3["gain"]]),
                      widgets.HBox([w3["Vpk"], w3["f_kHz"], w3["wave"]])]),
        widgets.interactive_output(draw_gbw_slew, w3))

## Leak four — the DC errors that decide precision

An ideal op-amp outputs zero for zero input. A real one has a few millivolts of **input offset** $V_{os}$ — the mismatch of the input pair from the previous notebook — and it draws a small **bias current** $I_B$ at each input.

Both are amplified by the *noise gain* $1+R_2/R_1$, regardless of which configuration you built:

$$V_{out,error}=V_{os}\left(1+\frac{R_2}{R_1}\right)+I_BR_2\ \ (\text{uncompensated})$$

With $V_{os}=2$ mV a gain of 100 puts **200 mV** of error at the output before any signal arrives. For a DC measurement that is fatal; for an audio stage it is irrelevant because a coupling capacitor removes it. Which of those you are building decides whether this section matters at all.

The bias-current term depends on the resistor values, which is why it is easy to miss. At $R_1=1$ kΩ, $R_2=10$ kΩ it contributes 91 µV — nothing. Raise them to 100 kΩ and 1 MΩ for the same gain and it becomes **9.1 mV**, larger than the offset. Adding a balancing resistor $R_1\parallel R_2$ on the other input makes the two bias currents cancel, leaving only their *difference* $I_{os}$, which is typically 5–10× smaller.

CMRR is the third. At 80 dB, a 1 V common-mode shift injects 100 µV of apparent input — invisible in most circuits and the dominant error in a high-side current sense.

In [ ]:
def draw_dc_errors(Vos_mV, Ib_nA, Ios_nA, R1_k, R2_k, balanced, cmrr_dB, Vcm):
    Vos, Ib, Ios = Vos_mV * 1e-3, Ib_nA * 1e-9, Ios_nA * 1e-9
    R1, R2 = R1_k * 1e3, R2_k * 1e3
    ng = 1 + R2 / R1
    e_os = Vos * ng
    e_ib = (Ios * R2) if balanced else (Ib * R2)
    e_cm = Vcm * 10 ** (-cmrr_dB / 20) * ng
    total = e_os + e_ib + e_cm
    rr = np.logspace(2, 6.5, 300)
    e_ib_curve = (Ios if balanced else Ib) * (rr * 10)

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.3, 0.55],
                          wspace=0.3, hspace=0.5, left=0.05, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 5.2), (-1.4, 2.6))
    (pn, pp, po) = opamp_tri(a0, (2.4, 0.6), 1.5, 1.4)
    wire(a0, [(0.4, 1.0), (1.1, 1.0)], 0.0, 1.0)
    resistor(a0, (1.1, 1.0), (2.0, 1.0), 0.0, 1.0, f"R1")
    wire(a0, [(2.0, 1.0), pn], 0.0, 1.0)
    wire(a0, [pn, (2.0, 2.2)], 0.0, 1.0)
    resistor(a0, (2.0, 2.2), (4.4, 2.2), 0.0, 1.0, f"R2")
    wire(a0, [(4.4, 2.2), (4.4, 0.6), po], total / 5, 1.0)
    if balanced:
        wire(a0, [pp, (1.9, 0.2)], 0.0, 1.0)
        resistor(a0, (1.9, 0.2), (1.9, -0.8), 0.0, 1.0, "R1||R2")
        wire(a0, [(1.9, -0.8), (1.9, -1.2), (0.6, -1.2)], 0.0, 1.0)
    else:
        wire(a0, [pp, (1.7, 0.2), (1.7, -1.2), (0.6, -1.2)], 0.0, 1.0)
    a0.annotate("", xy=(2.15, 1.0), xytext=(2.15, 1.55),
                arrowprops=dict(arrowstyle="-|>", color=NEG, lw=1.5))
    a0.text(2.2, 1.62, f"Ib {Ib_nA:.0f} nA", color=NEG, fontsize=7)
    a0.text(2.55, 0.95, f"Vos {Vos_mV:.2f} mV", color=ORANGE, fontsize=7)
    wire(a0, [po, (5.0, 0.6)], total / 5, 1.0)
    node_dot(a0, (5.0, 0.6), total / 5, 1.0)
    a0.text(4.6, 0.05, f"{total*1e3:+.3f} mV", color=DOT, fontsize=8)
    a0.set_title(f"zero in, {total*1e3:+.3f} mV out — noise gain "
                 f"{ng:.1f}", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    parts = [("Vos × NG", e_os), ("bias current", e_ib), ("CMRR", e_cm)]
    a1.bar(range(3), [abs(p[1]) * 1e3 for p in parts],
           color=[ORANGE, NEG, PURP], width=0.55)
    for i, (nm, v) in enumerate(parts):
        a1.text(i, abs(v) * 1e3, f" {abs(v)*1e3:.4f}", ha="center", va="bottom",
                fontsize=7.5)
    a1.set_xticks(range(3))
    a1.set_xticklabels([p[0] for p in parts], fontsize=7.5)
    a1.set_ylabel("output error  (mV)")
    a1.set_title(f"total {abs(total)*1e3:.4f} mV at the output")

    a2 = panel(fig.add_subplot(gs[1, 1]), NEG)
    a2.loglog(rr / 1e3, Ib * rr * 10 * 1e3, color=NEG, lw=1.7,
              label="$I_B R_2$, unbalanced")
    a2.loglog(rr / 1e3, Ios * rr * 10 * 1e3, color=GREEN, lw=1.7,
              label="$I_{os} R_2$, balanced")
    a2.axhline(Vos * ng * 1e3, color=ORANGE, lw=1.2, ls="--", label="Vos × NG")
    a2.axvline(R1_k, color=FG, lw=1.0, ls=":")
    a2.set_xlabel("R1  (kΩ),  with R2 = 10·R1"); a2.set_ylabel("mV")
    a2.legend(fontsize=7)
    a2.set_title("bias current only matters when the resistors are large")

    readout(fig, 0.845, 0.88, [
        "SPECS", "─" * 26,
        f"Vos         {Vos_mV:>10.3f}mV",
        f"Ib          {Ib_nA:>10.1f}nA",
        f"Ios         {Ios_nA:>10.1f}nA",
        f"CMRR        {cmrr_dB:>10.1f}dB",
        "", "CIRCUIT", "─" * 26,
        f"R1          {R1_k:>10.2f}kΩ",
        f"R2          {R2_k:>10.2f}kΩ",
        f"noise gain  {ng:>10.3f}",
        f"balanced    {str(bool(balanced)):>14s}",
        f"Vcm         {Vcm:>10.2f}V",
        "", "OUTPUT ERROR", "─" * 26,
        f"from Vos    {e_os*1e3:>+10.4f}mV",
        f"from Ib/Ios {e_ib*1e3:>+10.4f}mV",
        f"from CMRR   {e_cm*1e3:>+10.4f}mV",
        f"total       {total*1e3:>+10.4f}mV",
        f"as input    {total/ng*1e6:>+10.2f}µV",
        "", "a coupling capacitor",
        "removes all of this —",
        "if you are allowed one",
    ], color=NEG if abs(total) > 0.05 else GREEN)
    footer(fig, "error = Vos·(1+R2/R1) + Ib·R2 + Vcm·10^(−CMRR/20)·NG   ·   "
                "balancing swaps Ib for Ios")
    plt.show()


w4 = dict(Vos_mV=widgets.FloatSlider(value=2.0, min=0.01, max=10, step=0.01,
                                     description="Vos (mV):", **SL),
          Ib_nA=widgets.FloatSlider(value=100, min=0.01, max=1000, step=1,
                                    description="Ib (nA):", **SL),
          Ios_nA=widgets.FloatSlider(value=20, min=0.01, max=200, step=1,
                                     description="Ios (nA):", **SL),
          R1_k=widgets.FloatSlider(value=1, min=0.1, max=1000, step=1,
                                   description="R1 (kΩ):", **SL),
          R2_k=widgets.FloatSlider(value=10, min=1, max=10000, step=10,
                                   description="R2 (kΩ):", **SL),
          balanced=widgets.Checkbox(value=False, description="balancing resistor",
                                    indent=False),
          cmrr_dB=widgets.FloatSlider(value=80, min=40, max=140, step=5,
                                      description="CMRR (dB):", **SL),
          Vcm=widgets.FloatSlider(value=1.0, min=0, max=10, step=0.5,
                                  description="Vcm (V):", **SL))
display(widgets.VBox([widgets.HBox([w4["Vos_mV"], w4["Ib_nA"], w4["Ios_nA"]]),
                      widgets.HBox([w4["R1_k"], w4["R2_k"], w4["balanced"]]),
                      widgets.HBox([w4["cmrr_dB"], w4["Vcm"]])]),
        widgets.interactive_output(draw_dc_errors, w4))

## Leak five — the loop can oscillate

Feedback assumed the returning signal opposes the input. Every pole adds phase lag, and if the loop accumulates 180° while the loop gain is still above one, the opposition becomes **reinforcement** and the circuit oscillates.

The margin is measured where $|A\beta|=1$:

$$\text{PM}=180°+\angle A\beta\big|_{|A\beta|=1}$$

The step response and the Bode plot are two views of the same number, and the panel computes both from the same two-pole loop. Simulated rather than approximated:

| phase margin | overshoot |
|---|---|
| 30° | 41.6% |
| 45° | 23.3% |
| 60° | **8.8%** |
| 76° | 0.0% |

60° is the usual target because it is where ringing becomes negligible without giving up much speed, and 76° is where overshoot disappears entirely.

Two things make this a live problem rather than a datasheet curiosity. **Capacitive load** adds a pole at the output, inside the loop — a follower driving a long cable can ring badly, which is why series output resistors exist. And low closed-loop gain is the *dangerous* case: a follower has $\beta=1$, the maximum possible loop gain, so it has the least margin. An op-amp that is stable at a gain of 10 may oscillate as a follower, which is what "unity-gain stable" on a datasheet is promising.

In [ ]:
def draw_stability(A0_dB, f1_Hz, f2_kHz, gain, CL_pF):
    A0 = 10 ** (A0_dB / 20)
    f1, f2b = f1_Hz, f2_kHz * 1e3
    beta = 1.0 / gain
    f3 = 1 / (2 * np.pi * 200 * max(CL_pF, 0.01) * 1e-12) if CL_pF > 0 else 1e12
    f2 = 1 / (1 / f2b + 1 / f3)
    pm, fc = phase_margin(A0, f1, f2, beta)
    t, y, zeta, wn, dc = closed_step(A0, f1, f2, beta)
    over = (y.max() - dc) / dc * 100
    ff = np.logspace(0, 9, 700)
    L = two_pole_loop(A0, f1, f2, beta, ff)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.12)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.semilogx(ff, db(L), color=POS, lw=1.7)
    a0.axhline(0, color=MUTED, lw=0.9, ls="--")
    a0.axvline(fc, color=DOT, lw=1.1, ls=":")
    a0.text(fc * 1.2, 20, f"{fc/1e3:.1f} kHz", color=DOT, fontsize=7.5)
    a0.set_ylim(-40, A0_dB + 5); a0.set_ylabel("|Aβ|  (dB)")
    a0.set_title("loop gain")

    a1 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    a1.semilogx(ff, np.degrees(np.angle(L)), color=ORANGE, lw=1.7)
    a1.axhline(-180, color=NEG, lw=1.0, ls="--")
    a1.axvline(fc, color=DOT, lw=1.1, ls=":")
    a1.plot([fc], [np.degrees(np.angle(two_pole_loop(A0, f1, f2, beta,
                                                     np.array([fc]))))[0]],
            "o", ms=7, color=DOT)
    a1.annotate("", xy=(fc, -180), xytext=(fc, -180 + pm),
                arrowprops=dict(arrowstyle="<->", color=GREEN, lw=1.6))
    a1.text(fc * 1.3, -180 + pm / 2, f"PM {pm:.1f}°", color=GREEN, fontsize=8.5)
    a1.set_ylim(-200, 5); a1.set_xlabel("frequency  (Hz)")
    a1.set_ylabel("phase  (deg)")
    a1.set_title("phase margin is measured where the gain crosses unity")

    a2 = panel(fig.add_subplot(gs[:, 1]), GREEN)
    a2.plot(t * 1e6, y / dc, color=GREEN if pm > 45 else NEG, lw=1.8)
    a2.axhline(1.0, color=MUTED, lw=0.9, ls="--")
    a2.axhline(1 + over / 100, color=DOT, lw=0.8, ls=":")
    a2.text(t[-1] * 1e6 * 0.55, 1 + over / 100 + 0.02,
            f"overshoot {over:.2f}%", color=DOT, fontsize=8)
    a2.set_xlabel("time  (µs)"); a2.set_ylabel("normalised output")
    a2.set_title(f"the same information as a step response")

    readout(fig, 0.845, 0.88, [
        "LOOP", "─" * 26,
        f"A0          {A0_dB:>10.1f}dB",
        f"pole 1      {f1_Hz:>10.2f}Hz",
        f"pole 2      {f2b/1e3:>10.2f}kHz",
        f"closed gain {gain:>10.1f}",
        f"β           {beta:>10.5f}",
        "", "CAP LOAD", "─" * 26,
        f"CL          {CL_pF:>10.1f}pF",
        f"adds a pole {f3/1e3:>10.2f}kHz",
        f"effective p2{f2/1e3:>10.2f}kHz",
        "", "MARGIN", "─" * 26,
        f"crossover   {fc/1e3:>10.2f}kHz",
        f"phase margin{pm:>10.2f}°",
        f"zeta        {zeta:>10.4f}",
        f"overshoot   {over:>10.2f}%",
        "", "REFERENCE", "─" * 26,
        "PM 30° → 41.6%",
        "PM 45° → 23.3%",
        "PM 60° →  8.8%",
        "PM 76° →  0.0%",
        "", "a follower has β=1 —",
        "the LEAST margin",
    ], color=NEG if pm < 45 else (GREEN if pm > 55 else ORANGE))
    footer(fig, f"PM = 180° + ∠Aβ at |Aβ|=1 = {pm:.1f}°   ·   "
                f"overshoot {over:.1f}%   ·   capacitive load adds a pole inside the loop")
    plt.show()


w5 = dict(A0_dB=widgets.FloatSlider(value=100, min=60, max=140, step=5,
                                    description="A0 (dB):", **SL),
          f1_Hz=widgets.FloatSlider(value=10, min=1, max=100, step=1,
                                    description="pole 1 (Hz):", **SL),
          f2_kHz=widgets.FloatSlider(value=1000, min=10, max=10000, step=10,
                                     description="pole 2 (kHz):", **SL),
          gain=widgets.FloatSlider(value=1, min=1, max=100, step=1,
                                   description="closed gain:", **SL),
          CL_pF=widgets.FloatSlider(value=0, min=0, max=2000, step=10,
                                    description="load C (pF):", **SL))
display(widgets.HBox([w5["A0_dB"], w5["f1_Hz"], w5["f2_kHz"], w5["gain"],
                      w5["CL_pF"]]),
        widgets.interactive_output(draw_stability, w5))

## Putting it to work — the Sallen-Key filter

The passive RC from the AC notebook gave one pole and 20 dB/decade, and cascading two of them did *not* give a clean two-pole response because each stage loaded the next. An op-amp fixes precisely that, and in doing so lets a single stage produce a complex pole pair:

$$H(s)=\frac{K\omega_0^2}{s^2+\frac{\omega_0}{Q}s+\omega_0^2},\qquad
\omega_0=\frac{1}{RC},\qquad Q=\frac{1}{3-K}$$

The buffer's positive feedback through the first capacitor supplies the peaking that a passive network cannot. $Q$ is set entirely by the amplifier's gain $K$: $K=1$ gives $Q=0.5$, $K=1.586$ gives the maximally flat $Q=0.707$, and $K=2$ gives $Q=1$ with 1.25 dB of peaking.

The alarming part is what happens as $K\to3$. $Q$ goes to infinity, and beyond it the poles cross into the right half plane — the filter becomes an **oscillator**. Drag $K$ up and watch the peak grow without bound; a 4% error in a gain-setting resistor near that point is the difference between a filter and a sine-wave generator.

That is the closing point of the notebook. Every leak covered here — finite gain, bandwidth, slew, offset, phase margin — is a way in which the two resistors do not fully determine the answer, and the honest design question is never "does the ideal formula work" but "how much loop gain do I have left at the frequency where it matters".

In [ ]:
def sallen_key(f, R, C, K):
    w0 = 1 / (R * C)
    Q = 1 / max(3 - K, 1e-6)
    s = 2j * np.pi * f
    return K * w0 ** 2 / (s ** 2 + (w0 / Q) * s + w0 ** 2), w0, Q


def draw_sallen(R_k, C_nF, K, f_kHz):
    R, C = R_k * 1e3, C_nF * 1e-9
    ff = np.logspace(0, 6, 700)
    H, w0, Q = sallen_key(ff, R, C, K)
    f0 = w0 / (2 * np.pi)
    Hn, _, _ = sallen_key(np.array([f_kHz * 1e3]), R, C, K)
    peak_db = db(np.array([np.max(np.abs(H) / K)]))[0]
    zeta = 1 / (2 * Q)
    poles = [-zeta * w0 + 1j * w0 * np.sqrt(max(1 - zeta ** 2, 0)),
             -zeta * w0 - 1j * w0 * np.sqrt(max(1 - zeta ** 2, 0))]
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.25, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 5.4), (-1.2, 2.8))
    (pn, pp, po) = opamp_tri(a0, (3.0, 0.6), 1.4, 1.3, invert_top=False)
    wire(a0, [(0.3, 0.6), (0.9, 0.6)], 1.0, vmax)
    resistor(a0, (0.9, 0.6), (1.8, 0.6), 0.8, vmax, "R")
    resistor(a0, (1.8, 0.6), (2.7, 0.6), 0.5, vmax, "R")
    wire(a0, [(2.7, 0.6), pp], 0.4, vmax)
    capacitor(a0, (2.3, 0.6), (2.3, -0.7), 0.3, vmax, "C")
    wire(a0, [(2.3, -0.7), (2.3, -1.0), (0.6, -1.0)], 0.0, vmax)
    wire(a0, [(1.8, 0.6), (1.8, 2.2)], 0.5, vmax)
    capacitor(a0, (1.8, 2.2), (4.7, 2.2), 0.4, vmax, "C")
    wire(a0, [(4.7, 2.2), (4.7, 0.6), po], 0.9, vmax)
    wire(a0, [pn, (2.6, 0.15), (2.6, -0.5)], 0.6, vmax)
    resistor(a0, (2.6, -0.5), (2.6, -1.0), 0.3, vmax, None)
    a0.text(4.35, 2.42, "positive feedback", color=NEG, fontsize=7.5, ha="center")
    node_dot(a0, (1.8, 0.6), 0.5, vmax); node_dot(a0, (5.1, 0.6), 0.9, vmax)
    wire(a0, [po, (5.1, 0.6)], 0.9, vmax)
    a0.text(5.15, 0.85, "out", color=FG, fontsize=8)
    a0.set_title(f"K = {K:.4f}  →  Q = {Q:.4f}", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    for kk in (1.0, 1.586, 2.0, 2.5):
        Hx, _, Qx = sallen_key(ff, R, C, kk)
        a1.semilogx(ff, db(Hx / kk), color=POS if abs(kk - K) < 0.03 else MUTED,
                    lw=1.9 if abs(kk - K) < 0.03 else 0.8,
                    alpha=1.0 if abs(kk - K) < 0.03 else 0.5)
    a1.semilogx(ff, db(H / K), color=POS, lw=1.9)
    a1.axvline(f0, color=DOT, lw=1.0, ls="--")
    a1.axvline(f_kHz * 1e3, color=FG, lw=1.0, ls=":")
    a1.axhline(-3.01, color=MUTED, lw=0.7, ls=":")
    a1.set_ylim(-50, max(peak_db + 6, 8)); a1.set_ylabel("|H/K|  (dB)")
    a1.set_title(f"f0 = {f0:,.1f} Hz  ·  peaking {peak_db:.2f} dB  ·  "
                 f"40 dB/decade")

    a2 = panel(fig.add_subplot(gs[1, 1]), PURP)
    th = np.linspace(np.pi / 2, 3 * np.pi / 2, 200)
    a2.plot(w0 * np.cos(th) / 1e3, w0 * np.sin(th) / 1e3, color=GRIDC, lw=1.0)
    for p_ in poles:
        a2.plot([p_.real / 1e3], [p_.imag / 1e3], "x", ms=12, mew=2.6,
                color=NEG if p_.real > 0 else PURP)
    a2.axvline(0, color=NEG, lw=1.2)
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.set_xlim(-1.4 * w0 / 1e3, 0.5 * w0 / 1e3)
    a2.set_ylim(-1.2 * w0 / 1e3, 1.2 * w0 / 1e3)
    a2.set_xlabel("real  (krad/s)"); a2.set_ylabel("imag  (krad/s)")
    a2.set_title("K → 3 pushes the poles onto the axis")

    readout(fig, 0.845, 0.90, [
        "COMPONENTS", "─" * 26,
        f"R           {R_k:>10.2f}kΩ",
        f"C           {C_nF:>10.2f}nF",
        f"K           {K:>10.4f}",
        "", "RESPONSE", "─" * 26,
        f"f0 = 1/2πRC {f0:>10.2f}Hz",
        f"Q = 1/(3−K) {Q:>10.4f}",
        f"zeta        {zeta:>10.4f}",
        f"peaking     {peak_db:>10.3f}dB",
        f"slope       {40:>10.0f}dB/dec",
        "", "AT MARKER", "─" * 26,
        f"f           {f_kHz:>10.2f}kHz",
        f"|H/K|       {db(np.abs(Hn)/K)[0]:>+10.3f}dB",
        "", "REFERENCE", "─" * 26,
        "K=1.000 → Q=0.500",
        "K=1.586 → Q=0.707",
        "K=2.000 → Q=1.000",
        "K=3.000 → OSCILLATES",
        "", f"margin to K=3: {3-K:>7.4f}",
    ], color=NEG if K > 2.85 else (ORANGE if Q > 1.5 else GREEN))
    footer(fig, f"Q = 1/(3−K)   ·   K = {K:.3f} gives Q = {Q:.3f}   ·   "
                f"at K = 3 the poles reach the imaginary axis")
    plt.show()


w6 = dict(R_k=widgets.FloatSlider(value=10, min=1, max=100, step=1,
                                  description="R (kΩ):", **SL),
          C_nF=widgets.FloatSlider(value=10, min=0.1, max=100, step=0.1,
                                   description="C (nF):", **SL),
          K=widgets.FloatSlider(value=1.586, min=1.0, max=2.98, step=0.002,
                                description="gain K:", **SL),
          f_kHz=widgets.FloatSlider(value=1.6, min=0.01, max=100, step=0.01,
                                    description="marker (kHz):", **SL))
display(widgets.HBox([w6["R_k"], w6["C_nF"], w6["K"], w6["f_kHz"]]),
        widgets.interactive_output(draw_sallen, w6))